# 03 · BTC Options & Implied Volatility Surface

**Purpose:** Model the BTC options market — implied volatility surface, Greeks, vol term structure, skew.

BTC options on [Deribit](https://www.deribit.com) are the most liquid in crypto:
- European-style (no early exercise)
- Cash-settled in BTC
- Quoted in implied volatility

**Data sources:**
- Deribit REST API (no auth): `/public/get_book_summary_by_currency`, `/public/get_historical_volatility`
- `src/models/black_scholes.py` for pricing, IV solving, Greeks

---

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src.data.deribit import (
    get_options_chain, get_futures_chain, get_historical_volatility, get_index_price
)
from src.models.black_scholes import (
    bs_price, bs_greeks, implied_vol, atm_iv,
    risk_reversal_25d, butterfly_25d
)
from src.data.yfinance_fetcher import fetch_ohlcv
from src.utils.plotting import (
    plot_iv_surface, plot_vol_smile, plot_term_structure,
    PLOTLY_TEMPLATE, BTC_ORANGE
)
from config import RISK_FREE_RATE

pd.options.display.float_format = '{:.4f}'.format
print('Setup complete.')

## 1. Fetch Current BTC Spot Price

In [ ]:
try:
    S = get_index_price('btc_usd')
    print(f'BTC/USD (Deribit index): ${S:,.0f}')
except Exception as e:
    # fallback to yfinance
    btc_ohlcv = fetch_ohlcv('BTC-USD', period='5d')
    S = float(btc_ohlcv['Close'].iloc[-1])
    print(f'BTC/USD (yfinance fallback): ${S:,.0f}')

r = RISK_FREE_RATE
print(f'Risk-free rate: {r*100:.1f}%')

## 2. Fetch Options Chain from Deribit

In [ ]:
options = get_options_chain(currency='BTC', use_cache=True)
print(f'Options loaded: {len(options)} instruments')
print(f'Expiry range: {options["dte"].min():.0f} – {options["dte"].max():.0f} DTE')
print(f'Strike range: ${options["strike"].min():,.0f} – ${options["strike"].max():,.0f}')
print(f'Total open interest: {options["open_interest"].sum():,.0f} BTC')

# Filter to reasonable range for surface visualization
opts = options[
    options['dte'].between(2, 365) &
    options['mark_iv'].between(5, 250) &
    options['open_interest'].gt(0)
].copy()

opts['moneyness'] = opts['strike'] / S
opts['T'] = opts['dte'] / 365.0

print(f'\nFiltered to {len(opts)} options with OI > 0 and reasonable IV')
opts.head()

## 3. Implied Volatility Surface (3D)

In [ ]:
fig = plot_iv_surface(opts, S=S, title=f'BTC Implied Volatility Surface (Spot: ${S:,.0f})')
fig.show()

## 4. ATM Implied Volatility Term Structure

In [ ]:
# Get ATM IV for each expiry (nearest strike to spot)
atm_ivs = {}
for dte, grp in opts.groupby('dte'):
    closest = (grp['strike'] - S).abs().idxmin()
    iv_val = grp.loc[closest, 'mark_iv']
    if pd.notna(iv_val):
        atm_ivs[int(dte)] = float(iv_val)

atm_iv_series = pd.Series(atm_ivs, name='atm_iv').sort_index()
print('ATM IV by DTE:')
print(atm_iv_series.to_string())

fig = plot_term_structure(atm_iv_series, title='BTC ATM Implied Volatility Term Structure')
fig.show()

## 5. Volatility Smile by Expiry

In [ ]:
# Plot vol smile for each of the nearest 4 expiries
expiry_dtes = sorted(opts['dte'].unique())[:6]  # first 6 expiries

from plotly.subplots import make_subplots
import math

ncols = 2
nrows = math.ceil(len(expiry_dtes) / ncols)
fig = make_subplots(rows=nrows, cols=ncols,
                    subplot_titles=[f'{int(d)}DTE' for d in expiry_dtes])

for idx, dte in enumerate(expiry_dtes):
    row = idx // ncols + 1
    col = idx % ncols + 1
    grp = opts[opts['dte'] == dte].copy()
    grp['moneyness'] = grp['strike'] / S

    for opt_type, color, name in [('C', BTC_ORANGE, 'Call'), ('P', '#3498DB', 'Put')]:
        sub = grp[grp['opt_type'] == opt_type].dropna(subset=['mark_iv']).sort_values('moneyness')
        if sub.empty:
            continue
        fig.add_trace(go.Scatter(
            x=sub['moneyness'], y=sub['mark_iv'],
            mode='lines+markers', name=f'{name} ({int(dte)}DTE)',
            line=dict(color=color, width=2), marker=dict(size=4),
            showlegend=(idx == 0),
        ), row=row, col=col)

    fig.add_vline(x=1.0, line_dash='dash', line_color='gray', row=row, col=col)

fig.update_layout(
    title='BTC Volatility Smile by Expiry',
    template=PLOTLY_TEMPLATE, height=nrows * 300,
    xaxis_title='Moneyness (K/S)', yaxis_title='IV (%)',
)
fig.show()

## 6. Vol Skew: 25-Delta Risk Reversal & Butterfly

- **Risk Reversal (RR)** = IV(25Δ call) − IV(25Δ put). Positive → calls more expensive → bullish skew.
- **Butterfly (BF)** = 0.5×(IV(25Δ call) + IV(25Δ put)) − IV(ATM). Measures smile curvature.

In [ ]:
from src.models.black_scholes import bs_delta

skew_data = []

for dte, grp in opts.groupby('dte'):
    if dte < 3:
        continue
    T = dte / 365.0
    calls = grp[grp['opt_type'] == 'C'].dropna(subset=['mark_iv', 'strike'])
    puts  = grp[grp['opt_type'] == 'P'].dropna(subset=['mark_iv', 'strike'])

    if calls.empty or puts.empty:
        continue

    # Calculate delta for each option
    calls = calls.copy()
    puts  = puts.copy()
    calls['delta'] = calls.apply(
        lambda x: bs_delta(S, x['strike'], T, r, x['mark_iv']/100, 'C'), axis=1)
    puts['delta'] = puts.apply(
        lambda x: abs(bs_delta(S, x['strike'], T, r, x['mark_iv']/100, 'P')), axis=1)

    # Find 25-delta strikes
    try:
        call_25d_iv = calls.iloc[(calls['delta'] - 0.25).abs().argsort()[:1]]['mark_iv'].values[0]
        put_25d_iv  = puts.iloc[(puts['delta']  - 0.25).abs().argsort()[:1]]['mark_iv'].values[0]
        # ATM IV
        atm_strike_idx = (grp['strike'] - S).abs().idxmin()
        atm_iv_val = grp.loc[atm_strike_idx, 'mark_iv']

        skew_data.append({
            'dte': dte,
            'rr_25d': call_25d_iv - put_25d_iv,
            'bf_25d': 0.5 * (call_25d_iv + put_25d_iv) - atm_iv_val,
            'atm_iv': atm_iv_val,
        })
    except (IndexError, KeyError):
        continue

skew_df = pd.DataFrame(skew_data).set_index('dte').sort_index()

fig = make_subplots(rows=2, cols=1,
                    subplot_titles=['25Δ Risk Reversal (Call − Put IV)', '25Δ Butterfly (Smile Curvature)'])

fig.add_trace(go.Bar(x=skew_df.index, y=skew_df['rr_25d'],
                      marker_color=[BTC_ORANGE if v >= 0 else '#3498DB' for v in skew_df['rr_25d'].fillna(0)],
                      name='25Δ RR'), row=1, col=1)
fig.add_trace(go.Bar(x=skew_df.index, y=skew_df['bf_25d'],
                      marker_color='#27AE60', name='25Δ BF'), row=2, col=1)

fig.update_layout(title='BTC Vol Skew Metrics by DTE', template=PLOTLY_TEMPLATE, height=500)
fig.update_xaxes(title_text='Days to Expiry')
fig.show()
print(skew_df)

## 7. Black-Scholes Greeks Calculator

In [ ]:
# Example: compute Greeks for an ATM 30-day call option
K = round(S / 1000) * 1000  # nearest $1000 strike
T = 30 / 365.0

# Get ATM IV for ~30d expiry from Deribit
near_30d_dte = min(skew_df.index, key=lambda x: abs(x - 30))
sigma = skew_df.loc[near_30d_dte, 'atm_iv'] / 100 if not skew_df.empty else 0.75

greeks_call = bs_greeks(S, K, T, r, sigma, 'C')
greeks_put  = bs_greeks(S, K, T, r, sigma, 'P')

print(f'\nBlack-Scholes Greeks for ATM {near_30d_dte}DTE options')
print(f'Spot: ${S:,.0f}  Strike: ${K:,.0f}  IV: {sigma*100:.1f}%  r: {r*100:.1f}%')
print(f'\n{"Metric":<12} {"Call":>12} {"Put":>12}')
print('-' * 38)
for key in ['price', 'delta', 'gamma', 'vega', 'theta', 'rho']:
    c_val = greeks_call[key]
    p_val = greeks_put[key]
    if key == 'price':
        print(f'{key:<12} ${c_val:>10,.0f} ${p_val:>10,.0f}')
    elif key == 'vega':
        print(f'{key:<12} ${c_val:>10,.0f} ${p_val:>10,.0f}  (per 1% vol)')
    elif key == 'theta':
        print(f'{key:<12} ${c_val:>10,.0f} ${p_val:>10,.0f}  (per day)')
    else:
        print(f'{key:<12} {c_val:>12.4f} {p_val:>12.4f}')

## 8. Open Interest by Strike (Gamma Exposure)

Large call OI concentrations above spot create gamma exposure that market makers must hedge —
contributing to upside acceleration. Large put OI below spot creates a 'gravity' effect.

In [ ]:
# Aggregate OI by strike across all expiries
oi_by_strike = opts.groupby(['strike', 'opt_type'])['open_interest'].sum().unstack(fill_value=0)
oi_by_strike = oi_by_strike.loc[
    (oi_by_strike.index >= S * 0.6) & (oi_by_strike.index <= S * 1.6)
]

fig = go.Figure()
if 'C' in oi_by_strike.columns:
    fig.add_trace(go.Bar(
        x=oi_by_strike.index, y=oi_by_strike['C'],
        name='Call OI', marker_color=BTC_ORANGE, opacity=0.8,
    ))
if 'P' in oi_by_strike.columns:
    fig.add_trace(go.Bar(
        x=oi_by_strike.index, y=-oi_by_strike['P'],
        name='Put OI (negative)', marker_color='#3498DB', opacity=0.8,
    ))

fig.add_vline(x=S, line_dash='dash', line_color='white',
               annotation_text=f'Spot ${S:,.0f}', annotation_position='top right')
fig.update_layout(
    title='BTC Options Open Interest by Strike (all expiries)',
    xaxis_title='Strike ($)', yaxis_title='OI (BTC)',
    barmode='overlay', template=PLOTLY_TEMPLATE, height=450,
)
fig.show()

## 9. Historical vs Implied Volatility (HV vs IV)

In [ ]:
# Deribit's historical volatility index
try:
    hist_vol = get_historical_volatility('BTC')
    
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=hist_vol.index, y=hist_vol['historical_vol'],
        name='Deribit 30d HV', mode='lines',
        line=dict(color='#3498DB', width=2),
    ))
    
    # Add ATM IV term structure point at 30d for comparison
    if near_30d_dte and not skew_df.empty:
        atm_30d = skew_df.loc[near_30d_dte, 'atm_iv']
        fig.add_hline(y=atm_30d, line_dash='dot', line_color=BTC_ORANGE,
                       annotation_text=f'Current 30d ATM IV: {atm_30d:.1f}%',
                       annotation_position='top right')
    
    fig.update_layout(
        title='Bitcoin: Historical Volatility vs Current ATM IV',
        yaxis_title='Annualized Volatility (%)', xaxis_title='Date',
        template=PLOTLY_TEMPLATE, height=400,
    )
    fig.show()
except Exception as e:
    print(f'Historical vol fetch failed: {e}')

## Summary

- BTC options on Deribit exhibit a **vol smile** with elevated IV for deep OTM options vs ATM
- **Term structure**: short-dated IV tends to spike around events; long-dated IV is more stable
- **Risk reversal** measures call vs put premium: positive RR indicates bullish positioning skew
- Market makers delta-hedge options, creating spot price dynamics around large OI strikes

**Next:** [04 · Hashrate Derivatives & Hashprice →](./04_hashrate_derivatives_hashprice.ipynb)